In [2]:
import pandas as pd
import re
import nltk

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score

In [4]:
# If reviews.csv is in the same folder as this notebook
df = pd.read_csv("review.csv")

print("Dataset:")
print(df)

Dataset:
    id                                             review sentiment
0    1  I absolutely loved this movie, the story was b...  Positive
1    2  This film was amazing and the acting was excel...  Positive
2    3  I really enjoyed the movie, every scene was in...  Positive
3    4  The story was wonderful and the characters wer...  Positive
4    5  This was a fantastic film with great acting an...  Positive
5    6  I loved the emotional story and the brilliant ...  Positive
6    7  The movie was entertaining, exciting and very ...  Positive
7    8  An excellent film with a powerful story and ta...  Positive
8    9     I enjoyed every minute of this wonderful movie  Positive
9   10  The film was impressive, enjoyable and beautif...  Positive
10  11  I hated this movie, the story was boring and d...  Negative
11  12  This film was terrible and the acting was very...  Negative
12  13  I did not enjoy the movie, the scenes were boring  Negative
13  14  The story was awful and the cha

In [5]:
nltk.download('punkt')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\SIC\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\SIC\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [6]:
stop_words = set(stopwords.words('english'))

def preprocess(text):
    # Convert to lowercase
    text = text.lower()

    # Remove punctuation and special characters
    text = re.sub(r'[^a-zA-Z\s]', '', text)

    # Tokenize
    tokens = word_tokenize(text)

    # Remove stop words
    tokens = [word for word in tokens if word not in stop_words]

    # Convert tokens back to text
    return ' '.join(tokens)

df['cleaned_review'] = df['review'].apply(preprocess)

print("Preprocessed Reviews:")
print(df[['review', 'cleaned_review']])

Preprocessed Reviews:
                                               review  \
0   I absolutely loved this movie, the story was b...   
1   This film was amazing and the acting was excel...   
2   I really enjoyed the movie, every scene was in...   
3   The story was wonderful and the characters wer...   
4   This was a fantastic film with great acting an...   
5   I loved the emotional story and the brilliant ...   
6   The movie was entertaining, exciting and very ...   
7   An excellent film with a powerful story and ta...   
8      I enjoyed every minute of this wonderful movie   
9   The film was impressive, enjoyable and beautif...   
10  I hated this movie, the story was boring and d...   
11  This film was terrible and the acting was very...   
12  I did not enjoy the movie, the scenes were boring   
13  The story was awful and the characters were co...   
14  This was a horrible film with weak acting and ...   
15  I disliked the emotional story and the disappo...   
16     Th

In [7]:
X = df['cleaned_review']
y = df['sentiment']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

print('Training samples:', len(X_train))
print('Testing samples:', len(X_test))

Training samples: 14
Testing samples: 6


In [8]:
bow_vectorizer = CountVectorizer()

X_train_bow = bow_vectorizer.fit_transform(X_train)
X_test_bow = bow_vectorizer.transform(X_test)

print('BOW training matrix shape:', X_train_bow.shape)
print('BOW testing matrix shape:', X_test_bow.shape)

BOW training matrix shape: (14, 41)
BOW testing matrix shape: (6, 41)


In [9]:
vocabulary = bow_vectorizer.get_feature_names_out()

print('BOW Vocabulary:')
print(vocabulary)

BOW Vocabulary:
['absolutely' 'acting' 'actors' 'amazing' 'awful' 'beautiful' 'boring'
 'brilliant' 'characters' 'directed' 'direction' 'disappointing'
 'disliked' 'emotional' 'engaging' 'enjoy' 'enjoyed' 'every' 'excellent'
 'fantastic' 'film' 'frustrating' 'great' 'hated' 'loved' 'memorable'
 'minute' 'movie' 'performances' 'poor' 'poorly' 'powerful' 'regretted'
 'scenes' 'story' 'talented' 'terrible' 'unconvincing' 'watching' 'weak'
 'wonderful']


In [10]:
bow_model = MultinomialNB()
bow_model.fit(X_train_bow, y_train)

print('BOW Naive Bayes model trained successfully.')

BOW Naive Bayes model trained successfully.


In [11]:
print('Class Prior Probabilities:')

for class_name, log_probability in zip(
    bow_model.classes_,
    bow_model.class_log_prior_
):
    probability = 2.718281828 ** log_probability
    print(class_name, ':', probability)

Class Prior Probabilities:
Negative : 0.5000000000585271
Positive : 0.5000000000585271


In [12]:
feature_log_prob = pd.DataFrame(
    bow_model.feature_log_prob_,
    columns=bow_vectorizer.get_feature_names_out(),
    index=bow_model.classes_
)

print('Feature Log Probabilities:')
print(feature_log_prob)

Feature Log Probabilities:
          absolutely    acting    actors   amazing     awful  beautiful  \
Negative   -4.317488 -3.624341 -3.624341 -4.317488 -3.624341  -4.317488   
Positive   -3.637586 -3.232121 -3.637586 -3.637586 -4.330733  -3.637586   

            boring  brilliant  characters  directed  ...  powerful  regretted  \
Negative -2.931194  -4.317488   -4.317488 -3.624341  ... -4.317488  -3.624341   
Positive -4.330733  -3.637586   -3.637586 -4.330733  ... -3.637586  -4.330733   

            scenes     story  talented  terrible  unconvincing  watching  \
Negative -3.624341 -2.931194 -4.317488 -3.218876     -3.624341 -3.624341   
Positive -4.330733 -2.721295 -3.637586 -4.330733     -4.330733 -4.330733   

              weak  wonderful  
Negative -3.624341  -4.317488  
Positive -4.330733  -3.232121  

[2 rows x 41 columns]


In [13]:
bow_predictions = bow_model.predict(X_test_bow)

print('BOW Test Predictions:')

for review, actual, predicted in zip(X_test, y_test, bow_predictions):
    print('\nReview:', review)
    print('Actual:', actual)
    print('Predicted:', predicted)

BOW Test Predictions:

Review: horrible film weak acting poor direction
Actual: Negative
Predicted: Negative

Review: film impressive enjoyable beautifully directed
Actual: Positive
Predicted: Negative

Review: story awful characters completely forgettable
Actual: Negative
Predicted: Positive

Review: movie entertaining exciting well made
Actual: Positive
Predicted: Negative

Review: really enjoyed movie every scene interesting
Actual: Positive
Predicted: Positive

Review: movie boring confusing badly made
Actual: Negative
Predicted: Negative


In [14]:
bow_accuracy = accuracy_score(y_test, bow_predictions)

print('BOW + Naive Bayes Accuracy:', bow_accuracy)
print('BOW + Naive Bayes Accuracy (%):', bow_accuracy * 100)

BOW + Naive Bayes Accuracy: 0.5
BOW + Naive Bayes Accuracy (%): 50.0


In [15]:
new_reviews = [
    'The movie was amazing and wonderful',
    'The film was boring and terrible',
    'I enjoyed the brilliant acting'
]

new_reviews_cleaned = [preprocess(review) for review in new_reviews]
new_reviews_bow = bow_vectorizer.transform(new_reviews_cleaned)
new_predictions_bow = bow_model.predict(new_reviews_bow)

print('New / Unseen Reviews - BOW:')

for review, prediction in zip(new_reviews, new_predictions_bow):
    print('Review:', review)
    print('Predicted:', prediction)
    print()

New / Unseen Reviews - BOW:
Review: The movie was amazing and wonderful
Predicted: Positive

Review: The film was boring and terrible
Predicted: Negative

Review: I enjoyed the brilliant acting
Predicted: Positive



In [16]:
tfidf_vectorizer = TfidfVectorizer()

X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_test_tfidf = tfidf_vectorizer.transform(X_test)

print('TF-IDF training matrix shape:', X_train_tfidf.shape)
print('TF-IDF testing matrix shape:', X_test_tfidf.shape)

TF-IDF training matrix shape: (14, 41)
TF-IDF testing matrix shape: (6, 41)


In [17]:
tfidf_model = MultinomialNB()
tfidf_model.fit(X_train_tfidf, y_train)

print('TF-IDF Naive Bayes model trained successfully.')

TF-IDF Naive Bayes model trained successfully.


In [18]:
tfidf_predictions = tfidf_model.predict(X_test_tfidf)

print('TF-IDF Test Predictions:')

for review, actual, predicted in zip(X_test, y_test, tfidf_predictions):
    print('\nReview:', review)
    print('Actual:', actual)
    print('Predicted:', predicted)

TF-IDF Test Predictions:

Review: horrible film weak acting poor direction
Actual: Negative
Predicted: Negative

Review: film impressive enjoyable beautifully directed
Actual: Positive
Predicted: Negative

Review: story awful characters completely forgettable
Actual: Negative
Predicted: Positive

Review: movie entertaining exciting well made
Actual: Positive
Predicted: Negative

Review: really enjoyed movie every scene interesting
Actual: Positive
Predicted: Positive

Review: movie boring confusing badly made
Actual: Negative
Predicted: Negative


In [19]:
tfidf_accuracy = accuracy_score(y_test, tfidf_predictions)

print('TF-IDF + Naive Bayes Accuracy:', tfidf_accuracy)
print('TF-IDF + Naive Bayes Accuracy (%):', tfidf_accuracy * 100)

TF-IDF + Naive Bayes Accuracy: 0.5
TF-IDF + Naive Bayes Accuracy (%): 50.0


In [20]:
new_reviews_tfidf = tfidf_vectorizer.transform(new_reviews_cleaned)
new_predictions_tfidf = tfidf_model.predict(new_reviews_tfidf)

print('New / Unseen Reviews - TF-IDF:')

for review, prediction in zip(new_reviews, new_predictions_tfidf):
    print('Review:', review)
    print('Predicted:', prediction)
    print()

New / Unseen Reviews - TF-IDF:
Review: The movie was amazing and wonderful
Predicted: Positive

Review: The film was boring and terrible
Predicted: Negative

Review: I enjoyed the brilliant acting
Predicted: Positive



In [21]:
comparison = pd.DataFrame({
    'Review': new_reviews,
    'BOW Prediction': new_predictions_bow,
    'TF-IDF Prediction': new_predictions_tfidf
})

print('BOW vs TF-IDF Predictions:')
print(comparison)

BOW vs TF-IDF Predictions:
                                Review BOW Prediction TF-IDF Prediction
0  The movie was amazing and wonderful       Positive          Positive
1     The film was boring and terrible       Negative          Negative
2       I enjoyed the brilliant acting       Positive          Positive


In [22]:
accuracy_comparison = pd.DataFrame({
    'Method': [
        'BOW + Naive Bayes',
        'TF-IDF + Naive Bayes'
    ],
    'Accuracy': [
        bow_accuracy,
        tfidf_accuracy
    ],
    'Accuracy (%)': [
        bow_accuracy * 100,
        tfidf_accuracy * 100
    ]
})

print('Accuracy Comparison:')
print(accuracy_comparison)

Accuracy Comparison:
                 Method  Accuracy  Accuracy (%)
0     BOW + Naive Bayes       0.5          50.0
1  TF-IDF + Naive Bayes       0.5          50.0
